<a href="https://colab.research.google.com/github/AlexandreLouzada/exercicios-analise-dados/blob/master/web_dados_GABARITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌐 Explorando Dados na Web com Python — GABARITO

**Roteiro:** Nível 1 (requests GET, params, headers) → Nível 2 (JSON+deseção/erros, download binário) → Nível 3 (robots.txt, BeautifulSoup, CSV, `pd.read_html`).

**Colab:** `Arquivo > Fazer upload do notebook`. Bibliotecas: `requests`, `pandas`, `bs4` (BeautifulSoup) — no Colab, instale só o `bs4` se faltar: `!pip install beautifulsoup4`.

⚠️ Sempre use `timeout` para o código não travar, e respeite `robots.txt` na raspagem.

---

In [ ]:
import requests
import pandas as pd
import io
import re
from bs4 import BeautifulSoup

## 🟢 Nível 1 — URLs, Parâmetros e Cabeçalhos (Básico)

1.1 GET no JSONPlaceholder; 1.2 `params` (`userId=2` + `_limit=3`); 1.3 `headers` com User-Agent do projeto; 1.4 `status_code`, `resposta.url` e `timeout`.

In [ ]:
# 1.1 + 1.2 + 1.3 + 1.4
url_posts = "https://jsonplaceholder.typicode.com/posts"
params = {"userId": 2, "_limit": 3}
headers = {"User-Agent": "Projeto-WebDados/1.0"}

resposta = requests.get(url_posts, params=params, headers=headers, timeout=10)

# 1.4
print(f"Status HTTP: {resposta.status_code}")
print(f"URL final (com query string): {resposta.url}")

# Confirmar que veio JSON
dados_posts = resposta.json()
print(f"Quantidade retornada: {len(dados_posts)}")
print("Primeiro registro:")
print(dados_posts[0])

df_posts = pd.DataFrame(dados_posts)
df_posts

## 🟡 Nível 2 — JSON, Erros e Arquivos Binários (Intermediário)

### 2.1 Vários CEPs no ViaCEP → DataFrame

In [ ]:
ceps = ["01310100", "20040020", "90160000"]  # SP, RJ, RS
registros = []

for cep in ceps:
    url_cep = f"https://viacep.com.br/ws/{cep}/json/"
    r = requests.get(url_cep, timeout=10)
    r.raise_for_status()
    dados_cep = r.json()
    registros.append({
        "cep": dados_cep.get("cep"),
        "logradouro": dados_cep.get("logradouro"),
        "bairro": dados_cep.get("bairro"),
        "cidade": dados_cep.get("localidade"),
        "uf": dados_cep.get("uf")
    })

df_ceps = pd.DataFrame(registros)
df_ceps

### 2.2 Função de download segura com `try/except` + `raise_for_status()`

In [ ]:
def baixar_seguro(url, timeout=10):
    """Faz download com verificação de erros HTTP."""
    try:
        resposta = requests.get(url, timeout=timeout)
        resposta.raise_for_status()  # levanta erro se não for 2xx
        print(f"OK: {resposta.status_code} — {url}")
        return resposta
    except requests.exceptions.HTTPError:
        print(f"⚠️ Erro HTTP ao acessar {url} ({resposta.status_code})")
    except requests.exceptions.ConnectionError:
        print(f"⚠️ Não foi possível conectar em {url}")
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout ao acessar {url}")
    except requests.exceptions.RequestException:
        print(f"⚠️ Erro genérico ao acessar {url}")
    return None

# Teste com URL válida e com URL inválida
ok = baixar_seguro("https://viacep.com.br/ws/01310100/json/")
falha = baixar_seguro("https://viacep.com.br/ws/99999999/json/")  # CEP inexistente (200 com corpo vazio Many requests)
nada = baixar_seguro("https://httpbin.org/status/404")
print("\nRetornos:", "ok" if ok else "None", "|", falha, "|", nada)

### 2.3 Download de imagem (Picsum) em bytes → arquivo `.jpg`

In [ ]:
resposta_img = baixar_seguro("https://picsum.photos/400/400")
if resposta_img:
    with open("imagem_aleatoria.jpg", "wb") as arquivo:
        arquivo.write(resposta_img.content)
    print("Imagem salva: imagem_aleatoria.jpg")
    print("Tamanho (bytes):", len(resposta_img.content))
else:
    print("Download não realizado.")

## 🔵 Nível 3 — Webscraping e Ética (Avançado)

### 3.1 Verificar `robots.txt` antes de raspar

In [ ]:
site = "https://books.toscrape.com"
robots = requests.get(f"{site}/robots.txt", timeout=10)
print(f"robots.txt status: {robots.status_code}")
print("-------------")
print(robots.text[:1000])

### 3.2–3.3 Extrair 5 livros com BeautifulSoup → CSV

In [ ]:
resposta_site = requests.get(
    "https://books.toscrape.com/",
    headers={"User-Agent": "Projeto-WebDados/1.0, etica no scraping"},
    timeout=10
)
resposta_site.raise_for_status()

sopa = BeautifulSoup(resposta_site.text, "html.parser")

# Cada livro fica em <article class="product_pod">
artigos = sopa.select("article.product_pod")[:5]

livros = []
for artigo in artigos:
    titulo = artigo.h3.a.get("title")  # atributo title tem o nome completo
    preco_raw = artigo.select_one("p.price_color").get_text()
    preco = re.sub(r"[^\d.,]", "", preco_raw)  # extrai só números (limpa moeda/símbolos)
    livros.append({"titulo": titulo, "preco": preco})

df_livros = pd.DataFrame(livros)
df_livros

In [ ]:
df_livros.to_csv("livros_toscrape.csv", index=False, encoding="utf-8")
print("CSV salvo: livros_toscrape.csv")
print(pd.read_csv("livros_toscrape.csv"))

### 3.4 Capturar uma tabela da Wikipedia com `pd.read_html` + `io.StringIO`

In [ ]:
url_wiki = "https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o"
html_wiki = requests.get(url_wiki, headers={"User-Agent": "Projeto-WebDados/1.0"}, timeout=15).text

# 1º) variação anual/pop 
tabelas = pd.read_html(io.StringIO(html_wiki))
print("Tabelas encontradas:", len(tabelas))
df_wiki = tabelas[0]
df_wiki.head(10)

## 🏁 Checklist

- [ ] N1: GET + params + headers + status/url/timeout
- [ ] N2: ViaCEP → DataFrame; função segura com `raise_for_status`; imagem `.jpg` em bytes
- [ ] N3.1: robots.txt consultado
- [ ] N3.2–3: BeautifulSoup → 5 livros → CSV
- [ ] N3.4: `read_html` + `StringIO` → DataFrame

Boa análise! 🎉